# PlantMetWiki figures

Publication-quality figures for the PlantMetWiki paper.

---

## How to run

**Step 1 — generate data from Virtuoso** (run once per data version, ~30 sec):
```bash
conda activate plantmetwiki-rdf

# Make sure Virtuoso is running
# (cd to Snorql-UI: docker compose up -d virtuoso)

# Query Virtuoso and write CSVs to notebooks/figures/output/
python scripts/generate_figures_data.py

# Skip slow per-species query for quick iteration
python scripts/generate_figures_data.py --skip-species
```

**Step 2 — run this notebook** (loads CSVs, produces figures):
```
Kernel → Restart & Run All
```

Figures are saved to **`figures/output/figures/`** as `.pdf` and `.svg`.

---

## Architecture

```
Virtuoso (graph/pathways + graph/gpml-taxonomy-extra + graph/ncbitaxon)
    │
    ▼  scripts/generate_figures_data.py   (queries via SPARQLWrapper)
notebooks/figures/output/*.csv            ← committed to git
    │
    ▼  this notebook
notebooks/figures/output/figures/*.pdf/.svg
```

The CSVs are committed to git so **this notebook runs without Virtuoso**.
Only re-run `generate_figures_data.py` when the underlying data changes.

---

## Named graphs queried

| CSV | Graph(s) |
|---|---|
| `genes/metabolites/enzymes/conversions_per_pathway.csv` | `graph/pathways` |
| `interaction_types.csv` | `graph/pathways` |
| `pathway_titles.csv` | `graph/pathways` |
| `species_per_pathway.csv` | `graph/pathways` + `graph/gpml-taxonomy-extra` |
| `per_species_nrs.csv` | all three + `graph/ncbitaxon` (species labels) |

**Kernel:** `plantmetwiki-rdf`

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.lines import Line2D

# ── Paths ─────────────────────────────────────────────────────────────────────
OUT_DIR  = Path('figures/output')
FIG_DIR  = OUT_DIR / 'figures'   # PDF/SVG outputs go here
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 150, 'font.size': 10})

# ── Load pre-computed CSVs ────────────────────────────────────────────────────
def load(name):
    path = OUT_DIR / name
    assert path.exists(), f'Missing: {path}\nRun: python scripts/generate_figures_data.py'
    return pd.read_csv(path)

def save_fig(fig, name):
    for ext in ('pdf', 'svg'):
        fig.savefig(FIG_DIR / f'{name}.{ext}', bbox_inches='tight')
    print(f'  Saved → {FIG_DIR}/{name}.{{pdf,svg}}')

WP_INTERACTION = 'http://vocabularies.wikipathways.org/wp#Interaction'

print('Loading CSVs ...')
genes       = load('genes_per_pathway.csv')
metabolites = load('metabolites_per_pathway.csv')
enzymes     = load('enzymes_per_pathway.csv')
conversions = load('conversions_per_pathway.csv')
int_types   = load('interaction_types.csv')
per_species = load('per_species_nrs.csv')
species_pw  = load('species_per_pathway.csv')
titles      = load('pathway_titles.csv')
print(f'  pathways: {len(titles):,}  |  genes: {len(genes):,}  |  species: {len(per_species):,}')
print(f'  Figures will be saved to: {FIG_DIR.resolve()}')

---
## 2. Normalize

In [12]:
def norm(df, count_col, count_name):
    df = df.copy()
    df = df.rename(columns={count_col: count_name})
    df[count_name] = pd.to_numeric(df[count_name], errors="coerce").fillna(0)
    if "pwID" in df.columns:
        df["pwID"] = df["pwID"].astype(str)
    return df

genes       = norm(genes,       "count", "genes")
metabolites = norm(metabolites, "count", "metabolites")
enzymes     = norm(enzymes,     "count", "enzymes")
conversions = norm(conversions, "count", "conversions")
species_pw  = norm(species_pw,  "count", "species")

int_types["n"] = pd.to_numeric(int_types["n"], errors="coerce").fillna(0).astype(int)
WP_INTERACTION = "http://vocabularies.wikipathways.org/wp#Interaction"
total_interactions = int(int_types.loc[int_types["type"] == WP_INTERACTION, "n"].iloc[0])

print("✅ Data ready")
print(f"  Pathways with gene data:       {len(genes):,}")
print(f"  Pathways with metabolite data: {len(metabolites):,}")
print(f"  Total interactions:            {total_interactions:,}")
print(f"  Species in per-species table:  {len(per_species):,}")

IndexError: single positional indexer is out-of-bounds

---
## 3. Figure: Overview bar chart (log scale)

In [ ]:
overview = {
    "Pathways":     len(genes),
    "Genes":        int(genes["genes"].sum()),
    "Metabolites":  int(metabolites["metabolites"].sum()),
    "Enzymes":      int(enzymes["enzymes"].sum()),
    "Interactions": total_interactions,
}
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(list(overview.keys()), list(overview.values()))
ax.set_yscale("log")
ax.set_ylabel("Count (log scale)")
ax.set_title("PlantMetWiki — content overview")
for bar, val in zip(bars, overview.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.1,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_overview_barlog")
plt.show()

---
## 4. Figure: Cumulative coverage curves

In [ ]:
def cumulative_curve(df, col):
    s = df[col].sort_values(ascending=False)
    total = s.sum()
    return (s.cumsum()/total).values if total > 0 else s.cumsum().values

curves = {
    "Genes":           cumulative_curve(genes, "genes"),
    "Enzymes":         cumulative_curve(enzymes, "enzymes"),
    "Metabolites":     cumulative_curve(metabolites, "metabolites"),
    "Conversions":     cumulative_curve(conversions, "conversions"),
    "Species/pathway": cumulative_curve(species_pw, "species"),
}
styles = {
    "Genes":           dict(linestyle="-",  marker="o", markevery=100),
    "Enzymes":         dict(linestyle="--", marker="s", markevery=100),
    "Metabolites":     dict(linestyle="-.", marker="^", markevery=100),
    "Conversions":     dict(linestyle=":",  marker="x", markevery=100),
    "Species/pathway": dict(linestyle="--", marker="D", markevery=100),
}
fig, ax = plt.subplots(figsize=(7, 4.8))
for label, curve in curves.items():
    ax.plot(range(1, len(curve)+1), curve, label=label, linewidth=2, **styles[label])
ax.set_xlabel("Pathways (ranked by contribution)")
ax.set_ylabel("Cumulative fraction of total")
ax.set_ylim(0, 1.01)
ax.set_title("Cumulative coverage of PlantMetWiki content")
ax.legend()
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_cumulative_coverage")
plt.show()

---
## 5. Figure: Interaction types (2-panel)

In [ ]:
label_map = {
    "http://vocabularies.wikipathways.org/wp#DirectedInteraction":      "Directed interaction",
    "http://vocabularies.wikipathways.org/wp#Conversion":               "Biochemical conversion",
    "http://vocabularies.wikipathways.org/wp#Catalysis":                "Catalysis",
    "http://vocabularies.wikipathways.org/wp#TranscriptionTranslation": "Transcription/translation",
    "http://vocabularies.wikipathways.org/wp#Inhibition":               "Inhibition",
    "http://vocabularies.wikipathways.org/wp#Stimulation":              "Stimulation",
    "http://vocabularies.wikipathways.org/wp#Binding":                  "Binding",
}
sub = int_types[int_types["type"] != WP_INTERACTION].copy()
sub["label"] = sub["type"].map(label_map).fillna(sub["type"])
sub["pct"]   = 100 * sub["n"] / total_interactions
sub = sub.sort_values("n", ascending=True)

fig, (ax0, ax1) = plt.subplots(1, 2, figsize=(10.5, 4.2),
                                gridspec_kw={"width_ratios": [1, 3]})
ax0.bar(["Total\ninteractions"], [total_interactions])
ax0.set_ylabel("Count"); ax0.set_title("A")
ax0.text(0, total_interactions, f"{total_interactions:,}",
         ha="center", va="bottom", fontsize=10)
ax0.spines[["top","right"]].set_visible(False)

ax1.barh(sub["label"], sub["n"])
ax1.set_xlabel("Number of interactions"); ax1.set_title("B")
xmax = sub["n"].max()
for y, (n, pct) in enumerate(zip(sub["n"], sub["pct"])):
    ax1.text(n+xmax*0.01, y, f"{n:,}  ({pct:.1f}%)", va="center", fontsize=9)
ax1.set_xlim(0, xmax*1.25)
ax1.spines[["top","right"]].set_visible(False)

fig.suptitle("Interaction types in PlantMetWiki", y=1.02)
plt.tight_layout()
save_fig(fig, "plantmetwiki_interaction_types")
plt.show()

---
## 6. Figure: Species metrics — stacked bar (top 50)

In [ ]:
top50 = per_species.sort_values("pathways", ascending=False).head(50).copy()
stack_metrics = [c for c in ["genes","enzymes","metabolites"] if c in top50.columns]
COLORS = {"genes": "C1", "enzymes": "C2", "metabolites": "C3"}

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(top50))
bottom = np.zeros(len(top50))
for m in stack_metrics:
    ax.bar(x, top50[m].values, bottom=bottom, label=m.capitalize(), color=COLORS[m])
    bottom += top50[m].values
ax.set_xticks(x)
ax.set_xticklabels(top50["species"], rotation=60, ha="right", fontsize=8)
ax.set_ylabel("Count")
ax.set_title("PlantMetWiki content by species (top 50)")
ax.legend(ncol=3, frameon=False)
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
save_fig(fig, "plantmetwiki_species_metrics_stacked_top50")
plt.show()

---
## 6b. Figure: Extended species metrics — indirect via shared pathway

Genes/enzymes are **directly** annotated with `wp:organism` in the taxonomy-extra graph.
Metabolites, conversions, and publications have **no direct species tag** — they are
attributed *indirectly* via shared pathway: if a gene of species X is in pathway P,
all metabolites, conversions, and publications in P are counted for species X.

In [ ]:
top50_ext = per_species.sort_values('pathways', ascending=False).head(50).copy()

# Direct annotations (solid colours)
direct_metrics   = [c for c in ['genes', 'enzymes'] if c in top50_ext.columns]
# Indirect via pathway (hatched, lighter)
indirect_metrics = [c for c in ['metabolites', 'conversions', 'publications']
                    if c in top50_ext.columns]

COLORS_D = {'genes': 'C1', 'enzymes': 'C2'}
COLORS_I = {'metabolites': '#aec7e8', 'conversions': '#ffbb78', 'publications': '#98df8a'}

fig, ax = plt.subplots(figsize=(16, 5))
x = np.arange(len(top50_ext))
bottom = np.zeros(len(top50_ext))

for m in direct_metrics:
    ax.bar(x, top50_ext[m].values, bottom=bottom,
           label=f'{m.capitalize()} (direct)', color=COLORS_D[m])
    bottom += top50_ext[m].values

for m in indirect_metrics:
    ax.bar(x, top50_ext[m].values, bottom=bottom,
           label=f'{m.capitalize()} (indirect, pathway context)',
           color=COLORS_I[m], hatch='//', edgecolor='white', linewidth=0.3)
    bottom += top50_ext[m].values

ax.set_xticks(x)
ax.set_xticklabels(top50_ext['species'], rotation=60, ha='right', fontsize=7)
ax.set_ylabel('Count')
ax.set_title('PlantMetWiki: direct and pathway-context metrics by species (top 50)')
ax.legend(ncol=3, frameon=False, fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_fig(fig, 'plantmetwiki_species_extended_top50')
plt.show()

---
## 7. Figure: Scatter — genes vs metabolites (sized by species count)

In [ ]:
merged = metabolites[['pwID','metabolites']].merge(
    genes[['pwID','genes']], on='pwID', how='outer'
).merge(
    species_pw[['pwID','species']], on='pwID', how='outer'
).fillna(0)
merged = merged.merge(titles.rename(columns={'pwID':'pwID','title':'title'}),
                      on='pwID', how='left')
merged = merged[(merged['metabolites'] > 0) | (merged['genes'] > 0)].copy()
# Ensure species is numeric with no NaN
merged['species'] = pd.to_numeric(merged['species'], errors='coerce').fillna(0)

sp_min = int(merged['species'].min())
sp_max = int(merged['species'].max())
size = 6 + (merged['species'] - sp_min) / max(sp_max - sp_min, 1) * 154

bins   = sorted(set([0, 1, 2, 5, 10, max(10, sp_max)]))
blabels = [f"{bins[i-1]+1 if i>1 else 0}-{bins[i]}" if bins[i-1]+1 != bins[i]
           else str(bins[i]) for i in range(1, len(bins))]
merged['sp_bin']  = pd.cut(merged['species'], bins=bins, include_lowest=True,
                            right=True, labels=blabels)
merged['sp_code'] = merged['sp_bin'].cat.codes

fig, ax = plt.subplots(figsize=(6.8, 5.4))
ax.scatter(merged['metabolites'], merged['genes'],
           s=size, c=merged['sp_code'], cmap='viridis',
           alpha=0.35, edgecolors='none')
ax.set_xlabel('Metabolites per pathway')
ax.set_ylabel('Genes per pathway')
ax.set_title('Pathway content in PlantMetWiki')

cmap = plt.get_cmap('viridis')
norm_c = mpl.colors.Normalize(vmin=0, vmax=max(1, len(blabels)-1))
handles = [Line2D([0],[0], marker='o', linestyle='None',
                  markerfacecolor=cmap(norm_c(i)), markeredgecolor='none',
                  markersize=8, alpha=0.8) for i in range(len(blabels))]
ax.legend(handles, blabels, title='Species count', loc='best', frameon=False)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
save_fig(fig, 'scatter_genes_vs_metabolites_size_species')

merged.sort_values(['genes','metabolites'], ascending=False)[
    ['pwID','title','genes','metabolites','species']
].head(50).to_csv(OUT_DIR / 'top_pathways_by_genes.csv', index=False)
print('Saved top_pathways_by_genes.csv')
plt.show()

---
## 7b. Figure: Scatter with pathway labels for outliers

Labels the 15 most 'outlying' pathways — those furthest from the origin
(largest Euclidean distance from the point cloud centroid in genes–metabolites space).

In [ ]:
import numpy as np

# Reuse merged from the previous scatter cell
if 'merged' not in dir():
    merged = metabolites[['pwID','metabolites']].merge(
        genes[['pwID','genes']], on='pwID', how='outer'
    ).merge(species_pw[['pwID','species']], on='pwID', how='outer').fillna(0)
    merged = merged.merge(titles, on='pwID', how='left')
    merged = merged[(merged['metabolites'] > 0) | (merged['genes'] > 0)].copy()
    merged['species'] = pd.to_numeric(merged['species'], errors='coerce').fillna(0)

# Compute Euclidean distance from centroid (outlier score)
cx, cy = merged['metabolites'].mean(), merged['genes'].mean()
merged['_dist'] = np.sqrt((merged['metabolites'] - cx)**2 + (merged['genes'] - cy)**2)
outliers = merged.nlargest(15, '_dist')

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(merged['metabolites'], merged['genes'],
           s=20, alpha=0.25, color='steelblue', edgecolors='none', label='All pathways')
ax.scatter(outliers['metabolites'], outliers['genes'],
           s=60, color='tomato', edgecolors='white', linewidth=0.8,
           zorder=5, label='15 most outlying')

# Label each outlier, offsetting to avoid overlap
from adjustText import adjust_text  # pip install adjustText
texts = []
for _, row in outliers.iterrows():
    label = str(row.get('title', row['pwID'])).replace('superpathway of ', 's.p. ')
    if len(label) > 40:
        label = label[:38] + '…'
    texts.append(ax.text(row['metabolites'], row['genes'], label,
                         fontsize=6, color='#333'))
try:
    adjust_text(texts, ax=ax,
                arrowprops=dict(arrowstyle='-', color='#aaa', lw=0.5))
except Exception:
    pass  # adjustText is optional; labels still shown without adjustment

ax.set_xlabel('Metabolites per pathway')
ax.set_ylabel('Genes per pathway')
ax.set_title('Pathway content — 15 most outlying pathways labelled')
ax.legend(frameon=False)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
save_fig(fig, 'scatter_outlying_pathways_labelled')
plt.show()

---
## Sandbox